In [7]:
!pip install sentence-transformers

In [5]:
!pip install --upgrade transformers sentence-transformers

   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.8 MB 5.6 MB/s eta 0:00:02
   ---------- ----------------------------- 2.9/10.8 MB 10.5 MB/s eta 0:00:01
   -------------------- ------------------- 5.5/10.8 MB 10.8 MB/s eta 0:00:01
   ---------------------- ----------------- 6.0/10.8 MB 9.0 MB/s eta 0:00:01
   ------------------------ --------------- 6.6/10.8 MB 7.9 MB/s eta 0:00:01
   ---------------------------- ----------- 7.6/10.8 MB 6.9 MB/s eta 0:00:01
   ------------------------------ --------- 8.4/10.8 MB 6.3 MB/s eta 0:00:01
   --------------------------------- ------ 9.2/10.8 MB 5.9 MB/s eta 0:00:01
   ----------------------------------- ---- 9.7/10.8 MB 5.4 MB/s eta 0:00:01
   ------------------------------------- -- 10.2/10.8 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------- 10.8/10.8 MB 4.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transform

In [ ]:
import pandas as pd
df = pd.read_excel('behaviour_simulation_train.xlsx')

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')


embeddings = model.encode(df['content'].tolist(), show_progress_bar=True, convert_to_numpy=True)
print(embeddings.shape)
print(embeddings)

Batches:   0%|          | 0/9375 [00:00<?, ?it/s]

(300000, 384)
[[-0.03698686 -0.0214316   0.11477826 ... -0.00515599 -0.00155295
  -0.08958416]
 [-0.05627384 -0.08745366  0.03679999 ...  0.06174328 -0.00369589
  -0.05814923]
 [ 0.00012053  0.03061479 -0.02489772 ... -0.01552051  0.04840958
  -0.00649812]
 ...
 [-0.03110131 -0.01838608  0.01456277 ...  0.05102252 -0.05376612
   0.05286836]
 [-0.02998553 -0.00998587  0.00249651 ... -0.0716788  -0.09088013
   0.10556891]
 [ 0.03460236 -0.0127459  -0.05576075 ... -0.00344947  0.00224051
   0.00631142]]


In [ ]:
import csv


In [7]:
import sys
print(sys.getsizeof(embeddings)/1024/1024)

439.4532470703125


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import joblib
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# using all-MiniLM-L6-v2 embeddings model for embedding content
# adding embeddings to the dataframe

# using neural network as model to train


y = df['likes']

X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42)
embeddings_train, embeddings_test = train_test_split(embeddings, test_size=0.2, random_state=42)

# using log(likes) as target variable
y_train = np.log1p(y_train)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(df[['media_type', 'hour']])

X_train_final = encoder.transform(X_train[['media_type', 'hour']])

X_test_final = encoder.transform(X_test[['media_type', 'hour']])


username_avg_likes_train = X_train.groupby('username')['likes'].transform('mean').values.reshape(-1,1)
company_avg_likes_train = X_train.groupby('inferred company')['likes'].transform('mean').values.reshape(-1,1)

# username_avg_likes for x_test will be the username_avg_likes from the training set if username is in training set\
# else it will be 0 

username_avg_likes_test = X_test['username'].map(X_train.groupby('username')['likes'].mean()).fillna(0).values.reshape(-1,1)
company_avg_likes_test = X_test['inferred company'].map(X_train.groupby('inferred company')['likes'].mean()).fillna(0).values.reshape(-1,1)


X_train_final = np.hstack([
    X_train['char_count'].values.reshape(-1,1),
    X_train['word_count'].values.reshape(-1,1), 
    X_train['sentiment_polarity'].values.reshape(-1,1), 
    X_train['sentiment_subjectivity'].values.reshape(-1,1),
    X_train_final,  # encoded media_type
    username_avg_likes_train,
    company_avg_likes_train,
    embeddings_train  # embeddings for content
])

X_test_final = np.hstack([
    X_test['char_count'].values.reshape(-1,1),
    X_test['word_count'].values.reshape(-1,1),
    X_test['sentiment_polarity'].values.reshape(-1,1), 
    X_test['sentiment_subjectivity'].values.reshape(-1,1),
    X_test_final,  # encoded media_type
    username_avg_likes_test,
    company_avg_likes_test,
    embeddings_test  # embeddings for content
])


model = RandomForestRegressor()
model.fit(X_train_final, y_train)
preds = model.predict(X_test_final)
from sklearn.metrics import mean_squared_error

preds = np.expm1(preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print("RMSE:", rmse)


KeyError: "None of [Index(['media_type', 'hour'], dtype='object')] are in the [columns]"

I got RMSE as 4313 with Random Forest Regressor which I believe is not the best choice of model